# IndexTTS 2.5 — Colab WebUI

Run the Gradio WebUI on Google Colab (Python 3.12 + GPU).

**Steps**
1. Check GPU
2. Mount Drive (weights persist in `MyDrive/index-tts-cache`)
3. Clone `py3.12` and install
4. Download IndexTTS-2.5 if missing
5. `start()` — Cloudflare public link, Gradio share as fallback

Dub / highlight / intent pipelines stay in their own notebooks (see the last cell).


## 0. Check GPU

In [ ]:
import shutil
import subprocess
import sys

print(f"Python {sys.version}")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("WARNING: no GPU detected. CPU inference will be very slow.")


## 1. Mount Google Drive (model cache)

In [ ]:
import os
from pathlib import Path

CACHE_ROOT = "/content/drive/MyDrive/index-tts-cache"
LOCAL_FALLBACK = "/content/index-tts/checkpoints"

try:
    from google.colab import drive

    drive.mount("/content/drive")
    Path(CACHE_ROOT, "hf_home").mkdir(parents=True, exist_ok=True)
    Path(CACHE_ROOT, "torch_home").mkdir(parents=True, exist_ok=True)
    Path(CACHE_ROOT, "checkpoints-2.5").mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = f"{CACHE_ROOT}/hf_home"
    os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch_home"
    os.environ["INDEX_TTS_MODEL_DIR"] = f"{CACHE_ROOT}/checkpoints-2.5"
    print(f"Cache: {CACHE_ROOT}")
    print(f"INDEX_TTS_MODEL_DIR={os.environ['INDEX_TTS_MODEL_DIR']}")
except Exception as exc:
    os.environ["INDEX_TTS_MODEL_DIR"] = LOCAL_FALLBACK
    print(
        f"WARNING: Drive mount failed ({exc}). "
        f"Weights will not persist. Using {LOCAL_FALLBACK}"
    )


## 2. Clone repo and install

In [ ]:
import os

os.chdir("/content")
if os.path.isdir("/content/index-tts/.git"):
    os.chdir("/content/index-tts")
    get_ipython().system("git fetch origin")
    get_ipython().system("git checkout py3.12")
    get_ipython().system("git pull --ff-only origin py3.12")
else:
    get_ipython().system("git clone -b py3.12 https://github.com/deluxebear/index-tts.git")
    os.chdir("/content/index-tts")
print("cwd:", os.getcwd())


In [ ]:
import os

os.chdir("/content/index-tts")
get_ipython().system("bash tools/setup_colab.sh")
if int(get_ipython().user_ns.get("_exit_code", 0) or 0) != 0:
    raise SystemExit("pip install failed")


## 3. Download IndexTTS-2.5 checkpoints

In [ ]:
import os
from pathlib import Path

model_dir = os.environ["INDEX_TTS_MODEL_DIR"]
required = [
    "gpt.pth",
    "s2mel.pth",
    "codec.pth",
    "multilingual_zh_ja_yue_char_del.tiktoken",
    "wav2vec2bert_stats.pt",
]
missing = [name for name in required if not Path(model_dir, name).is_file()]
if missing:
    print(f"Downloading IndexTTS-2.5 (missing: {', '.join(missing)})...")
    get_ipython().system(
        f'huggingface-cli download IndexTeam/IndexTTS-2.5 --local-dir "{model_dir}"'
    )
    missing = [name for name in required if not Path(model_dir, name).is_file()]
    if missing:
        raise SystemExit(
            f"Download incomplete, still missing: {missing}. "
            "Download IndexTeam/IndexTTS-2.5 manually."
        )
else:
    print(f"Checkpoints already present at {model_dir}")


## 4. Start WebUI

In [ ]:
import os
import sys

sys.path.insert(0, "/content/index-tts")
from tools.colab import start

start(model_dir=os.environ["INDEX_TTS_MODEL_DIR"])


## 5. Other pipelines

These stay in their own notebooks (same Drive cache root):

- [DubbingPipeline_Colab.ipynb](https://github.com/deluxebear/index-tts/blob/py3.12/DubbingPipeline_Colab.ipynb)
- [HighlightPipeline_Colab.ipynb](https://github.com/deluxebear/index-tts/blob/py3.12/HighlightPipeline_Colab.ipynb)
- [IntentPipeline_Colab.ipynb](https://github.com/deluxebear/index-tts/blob/py3.12/IntentPipeline_Colab.ipynb)
